# 01 - Exploratory Data Analysis (EDA)

**Goal:** understand the data before modelling.

> **Data note:** this notebook runs on the **SYNTHETIC demo dataset** produced by `src/make_sample_data.py`. Every row is invented. Patterns below describe how the demo data was built, **not** the real Pakistani property market. When you switch to a real dataset, re-run the notebook and rewrite the observations.

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from the notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import SAMPLE_DATA_PATH, REPORTS_DIR, TARGET
from src.data_loader import load_csv
from src.data_preprocessing import clean_data, find_price_outliers
from src.eda import (describe_by_city, plot_price_distribution, plot_area_vs_price,
                     plot_price_by_city, plot_price_by_bedrooms, plot_correlation_heatmap)

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load and inspect

In [2]:
raw = load_csv(SAMPLE_DATA_PATH)
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
raw.head()

Shape: (1010, 8)
Columns: ['city', 'property_type', 'area_marla', 'bedrooms', 'bathrooms', 'floors', 'age_years', 'price_pkr']


,city,property_type,area_marla,bedrooms,bathrooms,floors,age_years,price_pkr
0,Peshawar,House,10,3,3,2,17.00,"30,340,000.00"
1,Lahore,Flat,20,7,8,2,0.00,"101,027,000.00"
2,Islamabad,Flat,5,2,2,1,26.00,"18,451,000.00"
3,Mardan,House,5,3,3,2,18.00,"11,178,000.00"
4,Mardan,House,10,5,5,2,30.00,"22,478,000.00"


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city           1010 non-null   str    
 1   property_type  1010 non-null   str    
 2   area_marla     1010 non-null   int64  
 3   bedrooms       1010 non-null   int64  
 4   bathrooms      1010 non-null   int64  
 5   floors         1010 non-null   int64  
 6   age_years      980 non-null    float64
 7   price_pkr      1010 non-null   float64
dtypes: float64(2), int64(4), str(2)
memory usage: 75.2 KB


In [4]:
print("Missing values per column:")
print(raw.isnull().sum())
print("\nFully duplicated rows:", raw.duplicated().sum())

Missing values per column:
city              0
property_type     0
area_marla        0
bedrooms          0
bathrooms         0
floors            0
age_years        30
price_pkr         0
dtype: int64

Fully duplicated rows: 10


**What we see:** 1,010 rows and 8 columns. Only `age_years` has missing values (30). There are 10 duplicate rows. Duplicates must go *before* the train/test split, otherwise the same house could appear in both sets and make the test score look better than it is.

## 2. Clean the data

In [5]:
df, report = clean_data(raw)
report

{'rows_before': 1010,
 'non_numeric_entries_set_to_nan': 0,
 'duplicates_removed': 10,
 'missing_or_nonpositive_price_removed': 0,
 'invalid_values_removed': 0,
 'rows_after': 1000,
 'missing_values_remaining': {'age_years': 30},
 'price_outliers_flagged_not_removed': 8}

- Exact duplicates were removed.
- No impossible values (negative area, 99 bedrooms) were found in this demo data. The code checks anyway because real scraped data usually has some.
- The 30 missing ages are **kept**. The scikit-learn pipeline fills them with the training median, so nothing leaks from the test set.
- Price outliers are **flagged, not deleted**. Next we look at them.

## 3. Descriptive statistics

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
area_marla,"1,000.00",7.85,4.72,3.00,5.00,7.00,10.00,20.00
bedrooms,"1,000.00",3.16,1.93,1.00,2.00,3.00,4.00,8.00
bathrooms,"1,000.00",3.14,1.96,1.00,2.00,3.00,4.00,8.00
floors,"1,000.00",1.69,0.64,1.00,1.00,2.00,2.00,3.00
age_years,970.00,19.53,11.30,0.00,10.00,19.00,29.00,39.00
price_pkr,"1,000.00","31,008,362.00","27,242,061.44","4,359,000.00","13,999,750.00","22,509,000.00","36,232,000.00","190,087,000.00"


In [7]:
describe_by_city(df)

,count,mean_price_million,median_price_million
city,,,
Islamabad,256,43.57,32.52
Lahore,253,36.57,27.37
Mardan,238,17.85,13.17
Peshawar,253,25.11,19.92


## 4. Price distribution

In [8]:
plot_price_distribution(df)

<Figure size 700x400 with 1 Axes>

**Observation:** most houses sit between about 14 and 36 million PKR (the middle 50%), with a long tail to the right up to roughly 190 million. A long right tail like this is normal for prices: a few very large, expensive houses pull the average (31.0 million) well above the median (22.5 million). This is why we report MAE and RMSE in PKR and do not trust the mean alone.

### Look at the outliers before deciding anything

In [9]:
outliers = df[find_price_outliers(df)].sort_values(TARGET, ascending=False)
outliers.head(8)

,city,property_type,area_marla,bedrooms,bathrooms,floors,age_years,price_pkr
979,Islamabad,House,20,8,8,3,4.00,"190,087,000.00"
343,Islamabad,House,20,8,8,2,12.00,"165,315,000.00"
607,Islamabad,House,20,8,8,2,18.00,"162,808,000.00"
701,Islamabad,House,20,8,7,2,8.00,"159,228,000.00"
419,Islamabad,House,20,8,7,2,NaN,"155,168,000.00"
858,Islamabad,House,20,8,8,2,14.00,"154,271,000.00"
374,Lahore,House,20,8,7,3,11.00,"152,979,000.00"
445,Islamabad,House,20,8,8,2,10.00,"152,064,000.00"


In [10]:
outliers[['area_marla', 'city', 'property_type']].describe(include='all')

,area_marla,city,property_type
count,8.00,8,8
unique,NaN,2,1
top,NaN,Islamabad,House
freq,NaN,7,8
mean,20.00,NaN,NaN
std,0.00,NaN,NaN
min,20.00,NaN,NaN
25%,20.00,NaN,NaN
50%,20.00,NaN,NaN
75%,20.00,NaN,NaN


**Decision:** 8 houses are flagged (about 152 to 190 million PKR). All 8 are 20-marla houses, 7 in Islamabad and 1 in Lahore, the biggest plots in the most expensive cities, which is consistent with how the demo data is built. They are real patterns, not errors, so **we keep them**. Deleting them would teach the model that big expensive houses do not exist. With a real dataset, you would instead check the original listing for typing mistakes (for example an extra zero).

## 5. Area vs price

In [11]:
plot_area_vs_price(df)

<Figure size 700x450 with 1 Axes>

**Observation:** price rises clearly with area, and the cities form separate bands: for the same area, Islamabad is highest and Mardan is lowest.

## 6. Price by city and bedrooms

In [12]:
plot_price_by_city(df)

<Figure size 700x450 with 1 Axes>

In [13]:
plot_price_by_bedrooms(df)

<Figure size 700x450 with 1 Axes>

**Observation:** median prices rank Islamabad (32.5 million) > Lahore (27.4) > Peshawar (19.9) > Mardan (13.2). Average price also rises with bedrooms, but remember bedrooms and area move together (bigger houses have more bedrooms), so bedrooms alone does not *cause* the higher price.

## 7. Correlation heatmap

In [14]:
plot_correlation_heatmap(df)

<Figure size 700x550 with 2 Axes>

**Observation:** price correlates most strongly with `area_marla` (about 0.85), then `bedrooms` (0.79) and `bathrooms` (0.75). Age has a weak negative link (about -0.13): older houses are slightly cheaper. Correlation between the inputs themselves is also high (area, bedrooms, bathrooms), which is fine for tree models and tolerable for Linear Regression here, but worth remembering when interpreting coefficients.

*Correlation only measures straight-line relationships between numbers. City and property type are text columns, so they are not in this chart, yet city clearly matters (section 6).*

## 8. Save the charts to `reports/`

In [15]:
from src.eda import save_all_eda_charts
[p.name for p in save_all_eda_charts(df, REPORTS_DIR)]

['eda_price_distribution.png',
 'eda_area_vs_price.png',
 'eda_price_by_city.png',
 'eda_price_by_bedrooms.png',
 'eda_correlation_heatmap.png']

## Summary

1. The data needed only light cleaning (duplicates, missing ages).
2. Prices are right-skewed; outliers are genuine large houses, so they stay.
3. Area, city and bedrooms are the strongest signals; age is weak.
4. Next: `python -m src.train` trains and compares the models.